<a href="https://colab.research.google.com/github/5heron/HeliosAI/blob/main/trial_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy pandas requests matplotlib seaborn scikit-learn pvlib xgboost lightgbm catboost openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


In [ ]:
"""
PV GLOBAL CLEAN PIPELINE V8 (physics-aware, refactored + visualizers)
=====================================================================

Learns:
    pv_per_kw = I_GEN_MAX / Apparent PV Size

as a function of:
    - Weather features available from typical APIs
    - Time-of-year / time-of-day features
    - Solar geometry (sun position)
    - Clear-sky irradiance + cloudiness proxy (SolarRad / cs_ghi)
    - POA irradiance (plane-of-array) under assumed tilt/azimuth
    - Simple physics-informed engineered features

Everything installation-specific (site, device, wiring) and
grid/inverter electrical measurements is removed.

Outputs:
    - train_global_v7.csv   (unchanged name for compatibility)
    - test_global_v7.csv

NEW (V8):
    - POA irradiance
    - safer solar interpolation + night masking
    - time-series CV helper
    - plots in ./plots:
        - loss_curve_<model>.png
        - preds_vs_actual_<model>.png
        - residuals_time_<model>.png
        - error_hist_<model>.png
        - regression_confusion_<model>.png
        - feature_importance_<model>.png
"""

import os
import zipfile
import requests
import re
import numpy as np
import pandas as pd

import pvlib
from pvlib.location import Location
from pvlib.irradiance import get_total_irradiance

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    ExtraTreesRegressor,
    BaggingRegressor,
    StackingRegressor,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# -------------------------------------------------------------------
# CONFIG / PATHS
# -------------------------------------------------------------------
PV_URL = (
    "https://data.london.gov.uk/download/2nlqm/"
    "81fb6b31-f6b2-4e12-b054-090319faec7b/PV%20Data.zip"
)
WX_URL = (
    "https://data.london.gov.uk/download/2nlqm/"
    "b4a7e790-8cb8-451c-b828-c4c5d8445705/Weather%20Data%202014-11-30.xlsx"
)

PV_ZIP = "PV_Data.zip"
PV_FOLDER = "PV_Data"
INNER_PV = os.path.join(PV_FOLDER, "PV Data - csv files only.zip")
INNER_FOLDER = os.path.join(PV_FOLDER, "CSV")
WX_FILE = "Weather_Data.xlsx"
DEVICE_XLS = os.path.join(
    PV_FOLDER,
    "deviceListTable with explanatory notes v2 - customer addresses removed.xlsx",
)

PV_CSV = os.path.join(
    INNER_FOLDER,
    "2014-11-28 Cleansed and Processed",
    "EXPORT HourlyData",
    "EXPORT HourlyData - Customer Endpoints.csv",
)

TARGET = "I_GEN_MAX"
PV_PER_KW_COL = "pv_per_kw"

# Weather features to keep (API-like)
WEATHER_FEATURES = [
    "SolarRad", "TempOut", "OutHum", "DewPt",
    "WindSpeed", "Rain", "RainRate", "Bar",
]

# Time & temp engineered features
ENGINEERED_FEATURES = [
    "DayOfYear", "Month", "Hour_sin", "Hour_cos",
    "Temp_Squared", "Dewpoint_Depression",
]

# Solar geometry & clear-sky + POA features
SOLAR_FEATURES = [
    "solar_zenith", "solar_azimuth", "solar_elevation",
    "cs_ghi", "cs_dni", "cs_dhi",
    "ghi_ratio",
    "poa_global", "poa_direct", "poa_diffuse", "poa_ratio",
]

# Default site parameters for London dataset
DEFAULT_LAT = 51.5
DEFAULT_LON = -0.1
DEFAULT_ALT = 50.0
DEFAULT_TZ = "Europe/London"

# Assumed PV plane for POA calculation (approximate typical UK roof)
PV_TILT_DEG = 30.0       # degrees
PV_AZIMUTH_DEG = 180.0   # 180° = south-facing

PLOTS_DIR = "plots"


# -------------------------------------------------------------------
# UTILS
# -------------------------------------------------------------------
def log_shape(df: pd.DataFrame, label: str) -> None:
    print(f"[{label}] Shape: {df.shape} Rows: {len(df):,}")


def ensure_plots_dir() -> None:
    if not os.path.exists(PLOTS_DIR):
        os.makedirs(PLOTS_DIR)


# -------------------------------------------------------------------
# DOWNLOAD
# -------------------------------------------------------------------
def download_data() -> None:
    """Download and extract PV + weather data if not already present."""
    if not os.path.exists(PV_ZIP):
        print("Downloading PV ZIP...")
        r = requests.get(PV_URL, timeout=60)
        r.raise_for_status()
        with open(PV_ZIP, "wb") as f:
            f.write(r.content)
        print("✓ PV ZIP downloaded.")

    if not os.path.exists(PV_FOLDER):
        print("Extracting PV ZIP...")
        with zipfile.ZipFile(PV_ZIP, "r") as z:
            z.extractall(PV_FOLDER)
        print("✓ PV extracted.")

    if not os.path.exists(INNER_FOLDER):
        print("Extracting inner PV CSV ZIP...")
        with zipfile.ZipFile(INNER_PV, "r") as z:
            z.extractall(INNER_FOLDER)
        print("✓ Inner PV CSV extracted.")

    if not os.path.exists(WX_FILE):
        print("Downloading Weather XLSX...")
        r = requests.get(WX_URL, timeout=60)
        r.raise_for_status()
        with open(WX_FILE, "wb") as f:
            f.write(r.content)
        print("✓ Weather downloaded.")


# -------------------------------------------------------------------
# LOAD + MERGE
# -------------------------------------------------------------------
def load_and_merge(tolerance: str = "30min") -> pd.DataFrame:
    """Load PV, weather, and devices; merge by time and serial number."""
    pv = pd.read_csv(PV_CSV)
    wx = pd.read_excel(WX_FILE)
    dev = pd.read_excel(DEVICE_XLS)

    # Build datetime as naive local timestamps (London time implied)
    pv["datetime"] = pd.to_datetime(
        pv["t_date"].astype(str) + " " + pv["t_time"].astype(str),
        errors="coerce",
    )
    wx["datetime"] = pd.to_datetime(
        wx["Date"].astype(str) + " " + wx["Time"].astype(str),
        errors="coerce",
    )

    pv = pv.dropna(subset=["datetime"]).sort_values("datetime")
    wx = wx.dropna(subset=["datetime"]).sort_values("datetime")

    dev["Serial Number"] = dev["Serial Number"].astype(str).str.strip()

    # Time-based nearest merge PV + weather
    df = pd.merge_asof(
        pv,
        wx,
        on="datetime",
        direction="nearest",
        tolerance=pd.Timedelta(tolerance),
    )

    # Device metadata (not used as features, but needed for size)
    df = df.merge(
        dev,
        left_on="SerialNo",
        right_on="Serial Number",
        how="left",
        validate="many_to_one",
    )
    df = df.drop(columns=["Serial Number"], errors="ignore")

    log_shape(df, "Post-merge")
    return df


# -------------------------------------------------------------------
# CLEAN + BASIC FILTERS
# -------------------------------------------------------------------
def clean_and_coerce(df: pd.DataFrame) -> pd.DataFrame:
    """Fix numeric types, enforce PV–SolarRad consistency, basic imputations."""
    numeric_cols = [
        "TempOut", "SolarRad", "OutHum", "DewPt",
        "WindSpeed", "Rain", "RainRate", "Bar",
    ]

    # Fix '---' and coerce to numeric
    for c in numeric_cols:
        if c in df.columns:
            col = df[c].replace("---", np.nan)
            df[c] = pd.to_numeric(col, errors="coerce")

    # Require target
    df = df.dropna(subset=[TARGET])

    # Physical sanity:
    # - If PV > 0, SolarRad must be known
    # - If PV == 0, SolarRad may be NaN → treat as 0 (night)
    if "SolarRad" in df.columns:
        before = len(df)
        keep_mask = df["SolarRad"].notna() | (df[TARGET] == 0)
        df = df[keep_mask].copy()
        dropped = before - len(df)
        print(f"[Filter] dropped {dropped:,}")

        df["SolarRad"] = df["SolarRad"].fillna(0.0)

    # Median-impute remaining weather NaNs
    for c in numeric_cols:
        if c in df.columns:
            med = df[c].median()
            df[c] = df[c].fillna(med)

    # Clean datetime
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df.dropna(subset=["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)

    log_shape(df, "[[Post-clean]]")
    return df


# -------------------------------------------------------------------
# DROP ELECTRICAL LEAKAGE + INDOOR / DERIVED
# -------------------------------------------------------------------
def drop_leakage_and_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Remove electrical leakage and non-forecastable/indoor fields."""
    leakage_patterns = [
        r"^I_", r"^P_", r"^Q_", r"^S_",
        r"^V_",
        r"^thd",
        r"f_min", r"f_max",
        r"Substation_.*Filtered",
        r"V_MAX_Rise_.*",
        r"I_GEN_MAX_Filtered",
    ]
    leak_cols = [
        c for c in df.columns
        if c != TARGET and any(re.search(p, c) for p in leakage_patterns)
    ]

    nonforecastable = [
        "HiTemp", "LowTemp", "HeatIndex", "WindChill",
        "THWIndex", "THSWIndex", "WindRun", "HiSpeed",
        "SolarEnergy", "HiSolarRad", "HeatD-D", "CoolD-D",
    ]
    indoor = [
        "InTemp", "InHum", "InDew", "InHeat", "InEMC",
        "InAirDensity", "ET", "WindSamp", "WindTx",
        "ISSRecept", "ArcInt",
    ]
    contextual = [
        "t_date", "t_time", "d_y", "d_m", "d_d", "d_w",
        "t_h", "t_m", "Date", "Time",
        "VA", "VB", "VC", "PA", "PB", "PC", "Substation_y",
    ]
    derived_cols = [c for c in nonforecastable + indoor + contextual if c in df.columns]

    to_drop = sorted(set(leak_cols + derived_cols))
    df = df.drop(columns=to_drop, errors="ignore")

    print(f"[Post-leakage drop] dropped {len(to_drop)} columns")
    log_shape(df, "[[Post-leakage drop]]")
    return df


# -------------------------------------------------------------------
# TIME / TEMP FEATURES
# -------------------------------------------------------------------
def add_time_temp_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add DayOfYear, Month, Hour_sin/cos, Temp_Squared, Dewpoint_Depression."""
    df["DayOfYear"] = df["datetime"].dt.dayofyear
    df["Month"] = df["datetime"].dt.month
    hour_float = df["datetime"].dt.hour + df["datetime"].dt.minute / 60.0

    df["Hour_sin"] = np.sin(2 * np.pi * hour_float / 24.0)
    df["Hour_cos"] = np.cos(2 * np.pi * hour_float / 24.0)

    if "TempOut" in df.columns:
        df["Temp_Squared"] = df["TempOut"] ** 2
    else:
        df["Temp_Squared"] = 0.0

    if "TempOut" in df.columns and "DewPt" in df.columns:
        df["Dewpoint_Depression"] = df["TempOut"] - df["DewPt"]
    else:
        df["Dewpoint_Depression"] = 0.0

    log_shape(df, "[[Post-time]]")
    return df


# -------------------------------------------------------------------
# SOLAR GEOMETRY + CLEAR-SKY + POA FEATURES
# (DST-safe: ambiguous/nonexistent → NaT, then masked)
# -------------------------------------------------------------------
def add_solar_features(
    df: pd.DataFrame,
    lat: float = DEFAULT_LAT,
    lon: float = DEFAULT_LON,
    alt: float = DEFAULT_ALT,
    tz: str = DEFAULT_TZ,
) -> pd.DataFrame:

    dt = pd.DatetimeIndex(df["datetime"])

    dt_local = dt.tz_localize(
        tz,
        ambiguous="NaT",
        nonexistent="NaT"
    )

    valid_mask = dt_local.notna()

    solar_cols = [
        "solar_zenith", "solar_azimuth", "solar_elevation",
        "cs_ghi", "cs_dni", "cs_dhi",
        "ghi_ratio",
        "poa_global", "poa_direct", "poa_diffuse", "poa_ratio",
    ]
    for col in solar_cols:
        df[col] = np.nan

    if valid_mask.any():
        site = Location(latitude=lat, longitude=lon, tz=tz, altitude=alt)
        times_valid = dt_local[valid_mask]

        solpos = site.get_solarposition(times=times_valid)
        cs = site.get_clearsky(times=times_valid, model="ineichen")

        zenith = solpos["zenith"].values
        azimuth = solpos["azimuth"].values

        df.loc[valid_mask, "solar_zenith"] = zenith
        df.loc[valid_mask, "solar_azimuth"] = azimuth
        df.loc[valid_mask, "solar_elevation"] = 90.0 - zenith

        df.loc[valid_mask, "cs_ghi"] = cs["ghi"].values
        df.loc[valid_mask, "cs_dni"] = cs["dni"].values
        df.loc[valid_mask, "cs_dhi"] = cs["dhi"].values

        # Cloudiness proxy
        cs_safe = df["cs_ghi"].replace(0, np.nan)
        df["ghi_ratio"] = df["SolarRad"] / cs_safe
        df["ghi_ratio"] = df["ghi_ratio"].clip(0, 2)

        # POA irradiance (simple fixed plane)
        poa = get_total_irradiance(
            surface_tilt=PV_TILT_DEG,
            surface_azimuth=PV_AZIMUTH_DEG,
            solar_zenith=df.loc[valid_mask, "solar_zenith"],
            solar_azimuth=df.loc[valid_mask, "solar_azimuth"],
            dni=df.loc[valid_mask, "cs_dni"],
            ghi=df.loc[valid_mask, "cs_ghi"],
            dhi=df.loc[valid_mask, "cs_dhi"],
        )

        df.loc[valid_mask, "poa_global"] = poa["poa_global"].values
        df.loc[valid_mask, "poa_direct"] = poa["poa_direct"].values
        df.loc[valid_mask, "poa_diffuse"] = poa["poa_diffuse"].values

        poa_safe = df["poa_global"].replace(0, np.nan)
        df["poa_ratio"] = df["SolarRad"] / poa_safe
        df["poa_ratio"] = df["poa_ratio"].clip(0, 2)

    # Sort and set index for time-based interpolation if needed
    df = df.sort_values("datetime").set_index("datetime")

    # For small gaps in solar/clear-sky series, interpolate in time
    df[solar_cols] = df[solar_cols].interpolate(
        method="time",
        limit_direction="both"
    )

    # Night masking: set irradiances to 0 when sun below horizon
    night_mask = df["solar_elevation"] <= 0
    for c in ["cs_ghi", "cs_dni", "cs_dhi", "poa_global", "poa_direct", "poa_diffuse"]:
        df.loc[night_mask, c] = 0.0

    # For ratios at night, set to 0 (no information)
    for c in ["ghi_ratio", "poa_ratio"]:
        df.loc[night_mask, c] = 0.0

    df = df.reset_index()
    df[solar_cols] = df[solar_cols].fillna(0)

    log_shape(df, "[[Post-solar]]")
    return df


# -------------------------------------------------------------------
# GLOBAL TRANSFORM (NORMALIZE + DROP IDENTITY)
# -------------------------------------------------------------------
def to_global_pv(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise PV to pv_per_kw and drop installation / wiring identity.
    """
    if "Apparent PV Size" not in df.columns:
        raise ValueError("Missing 'Apparent PV Size' column for normalization.")

    df = df[df["Apparent PV Size"] > 0].copy()
    df[PV_PER_KW_COL] = df[TARGET] / df["Apparent PV Size"]

    # Remove clearly impossible values
    df = df[(df[PV_PER_KW_COL] >= 0) & (df[PV_PER_KW_COL] <= 3.0)]

    # Drop size-related columns
    df = df.drop(columns=[c for c in ["Apparent PV Size", "Size in G83 Register"]
                          if c in df.columns])

    # Identity-like columns (not available from an API)
    identity_patterns = [
        r"^Serial", r"^Site", r"^Name", r"^Substation",
        r"^Feeder", r"^Port", r"^Comment", r"Postcode",
    ]
    id_cols = [
        c for c in df.columns
        if any(re.search(p, c) for p in identity_patterns)
    ]
    df = df.drop(columns=id_cols, errors="ignore")

    # Drop wind direction one-hots
    wind_cols = [c for c in df.columns
                 if c.startswith("WindDir_") or c.startswith("HiDir_")]
    df = df.drop(columns=wind_cols, errors="ignore")

    # Drop raw target
    df = df.drop(columns=[TARGET], errors="ignore")

    log_shape(df, "[[Post-global pv_per_kw]]")
    return df


# -------------------------------------------------------------------
# FINAL FEATURE SELECTION
# -------------------------------------------------------------------
def select_features(df: pd.DataFrame) -> pd.DataFrame:
    keep_cols = (
        ["datetime", PV_PER_KW_COL]
        + WEATHER_FEATURES
        + ENGINEERED_FEATURES
        + SOLAR_FEATURES
    )
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()
    log_shape(df, "[[Post-select]]")
    return df


# -------------------------------------------------------------------
# TRAIN/TEST SPLIT (TIME ORDERED)
# -------------------------------------------------------------------
def time_split(df: pd.DataFrame, test_size: float = 0.2):
    df = df.sort_values("datetime").reset_index(drop=True)
    n = len(df)
    cut = int(n * (1.0 - test_size))
    train = df.iloc[:cut].copy()
    test = df.iloc[cut:].copy()
    print(f"[Split] Train: {len(train):,}  Test: {len(test):,}")
    return train, test


# -------------------------------------------------------------------
# MASTER PIPELINE
# -------------------------------------------------------------------
def build_global_v7(
    test_size: float = 0.2,
    merge_tolerance: str = "30min",
    auto_download: bool = True,
    lat: float = DEFAULT_LAT,
    lon: float = DEFAULT_LON,
    alt: float = DEFAULT_ALT,
    tz: str = DEFAULT_TZ,
):
    """
    NOTE: kept original name build_global_v7 for compatibility,
    but implementation is V8 (with POA, extra features, etc.).
    """
    if auto_download:
        download_data()

    df = load_and_merge(tolerance=merge_tolerance)
    df = clean_and_coerce(df)
    df = drop_leakage_and_derived(df)
    df = add_time_temp_features(df)
    df = add_solar_features(df, lat=lat, lon=lon, alt=alt, tz=tz)
    df = to_global_pv(df)
    df = select_features(df)

    train_df, test_df = time_split(df, test_size=test_size)

    # Save
    train_df.to_csv("train_global_v7.csv", index=False)
    test_df.to_csv("test_global_v7.csv", index=False)

    print("\n✨ Global PV dataset V8 ready (weather + time + sun + POA → pv_per_kw).")
    print(
        "Final feature columns:",
        [c for c in train_df.columns if c not in ["datetime", PV_PER_KW_COL]],
    )
    return train_df, test_df


# -------------------------------------------------------------------
# OPTIONAL: LAG FEATURES
# -------------------------------------------------------------------
def add_lag_features(df: pd.DataFrame, lag_hours=(1, 2, 3)):
    """
    Add lagged versions of a subset of columns (weather only).
    Lag is done per time order, globally (not per-site).
    """
    df = df.sort_values("datetime").reset_index(drop=True)
    lag_cols = ["SolarRad", "TempOut", "OutHum", "DewPt"]

    for lag in lag_hours:
        for c in lag_cols:
            if c in df.columns:
                df[f"{c}_lag{lag}"] = df[c].shift(lag)

    # Drop early rows with NaNs introduced by lags
    df = df.dropna().reset_index(drop=True)
    return df


# -------------------------------------------------------------------
# VISUALIZATION HELPERS
# -------------------------------------------------------------------
def plot_loss_curve(model, X_train, y_train, X_test, y_test, model_name: str):
    """
    Plot loss curve for models that support staged_predict (GB, HGB).
    Saves to plots/loss_curve_<model>.png
    """
    if not hasattr(model, "staged_predict"):
        return

    ensure_plots_dir()

    train_losses = []
    test_losses = []

    for y_pred_train in model.staged_predict(X_train):
        train_losses.append(mean_squared_error(y_train, y_pred_train))

    for y_pred_test in model.staged_predict(X_test):
        test_losses.append(mean_squared_error(y_test, y_pred_test))

    n = len(train_losses)
    xs = np.arange(1, n + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(xs, train_losses, label="Train MSE")
    plt.plot(xs, test_losses, label="Test MSE")
    plt.xlabel("Iteration")
    plt.ylabel("MSE")
    plt.title(f"Loss Curve ({model_name})")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"loss_curve_{model_name}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[Plot] Saved loss curve → {path}")


def plot_predictions(y_true, y_pred, model_name: str):
    ensure_plots_dir()
    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=y_true, y=y_pred, s=10, alpha=0.4)
    max_val = max(np.max(y_true), np.max(y_pred))
    plt.plot([0, max_val], [0, max_val], "r--", linewidth=1)
    plt.xlabel("Actual pv_per_kw")
    plt.ylabel("Predicted pv_per_kw")
    plt.title(f"Predicted vs Actual ({model_name})")
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"preds_vs_actual_{model_name}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[Plot] Saved preds vs actual → {path}")


def plot_residuals_time(dates, y_true, y_pred, model_name: str):
    ensure_plots_dir()
    residuals = y_true - y_pred
    plt.figure(figsize=(10, 4))
    plt.plot(dates, residuals, linewidth=0.5)
    plt.axhline(0, color="red", linestyle="--", linewidth=1)
    plt.xlabel("Time")
    plt.ylabel("Residual (actual - pred)")
    plt.title(f"Residuals over time ({model_name})")
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"residuals_time_{model_name}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[Plot] Saved residuals time series → {path}")


def plot_error_histogram(y_true, y_pred, model_name: str):
    ensure_plots_dir()
    residuals = y_true - y_pred
    plt.figure(figsize=(8, 4))
    sns.histplot(residuals, kde=True, bins=50)
    plt.xlabel("Residual (actual - pred)")
    plt.title(f"Residual Histogram ({model_name})")
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"error_hist_{model_name}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[Plot] Saved residual histogram → {path}")


def plot_regression_confusion_matrix(y_true, y_pred, model_name: str):
    """
    "Confusion matrix" analogue for regression:
    2D density / heatmap of actual vs predicted.
    """
    ensure_plots_dir()
    plt.figure(figsize=(6, 5))
    hb = plt.hexbin(y_true, y_pred, gridsize=50, cmap="viridis", mincnt=1)
    plt.colorbar(hb, label="Count")
    max_val = max(np.max(y_true), np.max(y_pred))
    plt.plot([0, max_val], [0, max_val], "r--", linewidth=1)
    plt.xlabel("Actual pv_per_kw")
    plt.ylabel("Predicted pv_per_kw")
    plt.title(f"Regression Confusion Matrix ({model_name})")
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"regression_confusion_{model_name}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[Plot] Saved regression confusion matrix → {path}")


def plot_feature_importance(model, feature_names, model_name: str, top_n: int = 20):
    ensure_plots_dir()
    if not hasattr(model, "feature_importances_"):
        return

    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1][:top_n]
    imp_vals = importances[idx]
    imp_names = np.array(feature_names)[idx]

    plt.figure(figsize=(8, max(4, 0.3 * len(imp_names))))
    sns.barplot(x=imp_vals, y=imp_names)
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(f"Top {len(imp_names)} Feature Importances ({model_name})")
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"feature_importance_{model_name}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[Plot] Saved feature importance → {path}")


def generate_diagnostics(
    model,
    model_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols,
):
    """
    Generate diagnostics for a fitted model:
    - loss curve (if supported)
    - preds vs actual
    - residuals over time
    - residual histogram
    - 2D regression confusion matrix
    - feature importance (if supported)
    """
    # Use DataFrames for compatibility
    X_train = train_df[feature_cols]
    y_train = train_df[PV_PER_KW_COL]
    X_test = test_df[feature_cols]
    y_test = test_df[PV_PER_KW_COL]

    # Predictions
    y_pred = model.predict(X_test)

    # Loss curve (if applicable)
    plot_loss_curve(model, X_train, y_train, X_test, y_test, model_name)

    # Basic plots
    plot_predictions(y_test.values, y_pred, model_name)
    plot_residuals_time(test_df["datetime"].values, y_test.values, y_pred, model_name)
    plot_error_histogram(y_test.values, y_pred, model_name)
    plot_regression_confusion_matrix(y_test.values, y_pred, model_name)
    plot_feature_importance(model, feature_cols, model_name)


# -------------------------------------------------------------------
# TIME-SERIES CV HELPER (OPTIONAL)
# -------------------------------------------------------------------
def time_series_cv_score(
    df: pd.DataFrame,
    model,
    n_splits: int = 5,
    use_lags: bool = True,
):
    """
    Evaluate a single model with TimeSeriesSplit on the whole dataset.
    Returns list of RMSE and R2 scores across folds.
    """
    tmp = df.copy()
    if use_lags:
        tmp = add_lag_features(tmp)

    tmp = tmp.sort_values("datetime").reset_index(drop=True)
    feature_cols = [c for c in tmp.columns if c not in ["datetime", PV_PER_KW_COL]]

    X = tmp[feature_cols].values
    y = tmp[PV_PER_KW_COL].values

    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmses, r2s = [], []

    fold = 0
    for train_idx, val_idx in tscv.split(X):
        fold += 1
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        mdl = model
        mdl.fit(X_tr, y_tr)
        y_pred = mdl.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        rmses.append(rmse)
        r2s.append(r2)
        print(f"[CV] Fold {fold}  RMSE={rmse:.4f}  R2={r2:.4f}")

    return rmses, r2s


# -------------------------------------------------------------------
# MODELLING: TRAIN & COMPARE MODELS
# -------------------------------------------------------------------
def train_and_compare_models(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    use_lags: bool = True,
    make_plots: bool = True,
):
    """
    Train several tree-based models on the global features.
    Optionally add lag features before splitting into X/y.
    Generates diagnostics for the best model.
    """
    if use_lags:
        train_df = add_lag_features(train_df)
        test_df = add_lag_features(test_df)

    feature_cols = [c for c in train_df.columns if c not in ["datetime", PV_PER_KW_COL]]

    X_train = train_df[feature_cols]
    y_train = train_df[PV_PER_KW_COL]
    X_test = test_df[feature_cols]
    y_test = test_df[PV_PER_KW_COL]

    print(f"Train: {X_train.shape}  Test: {X_test.shape}")
    print("Features:", feature_cols)

    models = {
        "LGBM": LGBMRegressor(
            n_estimators=1200,
            learning_rate=0.02,
            max_depth=-1,
            num_leaves=256,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=20,
            reg_lambda=0.2,
            random_state=42,
        ),
        "XGB": XGBRegressor(
            n_estimators=1200,
            learning_rate=0.03,
            max_depth=8,
            subsample=0.85,
            colsample_bytree=0.8,
            tree_method="hist",
            reg_lambda=0.5,
            reg_alpha=0.2,
            eval_metric="rmse",
            random_state=42,
        ),
        "CatBoost": CatBoostRegressor(
            depth=8,
            learning_rate=0.03,
            iterations=1500,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            l2_leaf_reg=6,
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=600,
            max_depth=38,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        ),
        "RF": RandomForestRegressor(
            n_estimators=800,
            max_depth=28,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        ),
        "HGB": HistGradientBoostingRegressor(
            max_depth=10,
            learning_rate=0.04,
            max_iter=800,
            min_samples_leaf=20,
            random_state=42,
        ),
        "GB": GradientBoostingRegressor(
            n_estimators=900,
            learning_rate=0.025,
            max_depth=4,
            subsample=0.8,
            random_state=42,
        ),
        "Bagging": BaggingRegressor(
            n_estimators=50,
            oob_score=False,
            random_state=42,
            n_jobs=-1,
        ),
        "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.2, random_state=42),
        "SVR": SVR(kernel="rbf", C=2.5, epsilon=0.01, gamma="scale"),
        "KNN": KNeighborsRegressor(n_neighbors=5, weights="distance"),
        "MLP": MLPRegressor(
            hidden_layer_sizes=(200, 200),
            activation="relu",
            learning_rate_init=0.001,
            max_iter=800,
            random_state=42,
        ),
        "Stack": StackingRegressor(
            estimators=[
                (
                    "lgbm",
                    LGBMRegressor(
                        num_leaves=128,
                        learning_rate=0.05,
                        n_estimators=500,
                        random_state=42,
                    ),
                ),
                (
                    "xgb",
                    XGBRegressor(
                        max_depth=6,
                        n_estimators=500,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        random_state=42,
                    ),
                ),
                (
                    "rf",
                    RandomForestRegressor(
                        n_estimators=500,
                        max_depth=None,
                        random_state=42,
                    ),
                ),
            ],
            final_estimator=LGBMRegressor(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=128,
                random_state=42,
            ),
            passthrough=True,
            n_jobs=-1,
        ),
    }

    results = []
    fitted_models = {}

    for name, model in models.items():
        print(f"\n=== Training {name} ===")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rmse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        results.append({"model": name, "rmse": rmse, "r2": r2})
        fitted_models[name] = model
        print(f"    RMSE={rmse:.4f}  R2={r2:.4f}")

    results_df = pd.DataFrame(results).sort_values("rmse")
    print("\n=== Model comparison (sorted by RMSE) ===")
    print(results_df.to_string(index=False))

    best_name = results_df.iloc[0]["model"]
    best_model = fitted_models[best_name]
    print(f"\nBest model: {best_name}")

    if make_plots:
        generate_diagnostics(
            best_model,
            best_name,
            train_df=train_df,
            test_df=test_df,
            feature_cols=feature_cols,
        )

    return results_df, fitted_models


# -------------------------------------------------------------------
# MAIN
# -------------------------------------------------------------------
if __name__ == "__main__":
    # 1) Build dataset (physics-aware V8, but keep original function name)
    train_raw, test_raw = build_global_v7()

    # 2) Train models & compare, with visualizations for best model
    results_df, models = train_and_compare_models(
        train_raw,
        test_raw,
        use_lags=True,
        make_plots=True,
    )

    # OPTIONAL: time-series CV on best model (example)
    # from copy import deepcopy
    # best_name = results_df.iloc[0]["model"]
    # base_model = deepcopy(models[best_name])
    # print(f"\nRunning TimeSeriesSplit CV for best model: {best_name}")
    # rmses, r2s = time_series_cv_score(
    #     pd.concat([train_raw, test_raw], ignore_index=True),
    #     base_model,
    #     n_splits=5,
    #     use_lags=True,
    # )
    # print("CV RMSEs:", rmses)
    # print("CV R2s:  ", r2s)

✓ PV ZIP downloaded.
Extracting PV ZIP...
✓ PV extracted.
Extracting inner PV CSV ZIP...
✓ Inner PV CSV extracted.
✓ Weather downloaded.
[Post-merge] Shape: (52352, 117) Rows: 52,352
[Filter] dropped 9,490


/tmp/ipython-input-3476907161.py:242: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  col = df[c].replace("---", np.nan)


[[[Post-clean]]] Shape: (42862, 117) Rows: 42,862
[Post-leakage drop] dropped 91 columns
[[[Post-leakage drop]]] Shape: (42862, 26) Rows: 42,862
[[[Post-time]]] Shape: (42862, 31) Rows: 42,862
[[[Post-solar]]] Shape: (42862, 42) Rows: 42,862
[[[Post-global pv_per_kw]]] Shape: (37890, 33) Rows: 37,890
[[[Post-select]]] Shape: (37890, 27) Rows: 37,890
[Split] Train: 30,312  Test: 7,578

✨ Global PV dataset V8 ready (weather + time + sun + POA → pv_per_kw).
Final feature columns: ['SolarRad', 'TempOut', 'OutHum', 'DewPt', 'WindSpeed', 'Rain', 'RainRate', 'Bar', 'DayOfYear', 'Month', 'Hour_sin', 'Hour_cos', 'Temp_Squared', 'Dewpoint_Depression', 'solar_zenith', 'solar_azimuth', 'solar_elevation', 'cs_ghi', 'cs_dni', 'cs_dhi', 'ghi_ratio', 'poa_global', 'poa_direct', 'poa_diffuse', 'poa_ratio']
Train: (30309, 37)  Test: (7575, 37)
Features: ['SolarRad', 'TempOut', 'OutHum', 'DewPt', 'WindSpeed', 'Rain', 'RainRate', 'Bar', 'DayOfYear', 'Month', 'Hour_sin', 'Hour_cos', 'Temp_Squared', 'Dewpoi

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.410e+03, tolerance: 2.312e+00
  model = cd_fast.enet_coordinate_descent(


    RMSE=0.5754  R2=0.2154

=== Training SVR ===
    RMSE=0.8164  R2=-0.1132

=== Training KNN ===
    RMSE=0.6484  R2=0.1159

=== Training MLP ===
    RMSE=1.4226  R2=-0.9398

=== Training Stack ===
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038853 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7731
[LightGBM] [Info] Number of data points in the train set: 30309, number of used features: 40
[LightGBM] [Info] Start training from score 1.101194


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    RMSE=0.5850  R2=0.2023

=== Model comparison (sorted by RMSE) ===
     model     rmse        r2
  CatBoost 0.572953  0.218751
        GB 0.574114  0.217169
ElasticNet 0.575428  0.215376
       HGB 0.576740  0.213588
        RF 0.577006  0.213225
     Stack 0.585029  0.202286
ExtraTrees 0.586470  0.200321
   Bagging 0.588513  0.197534
       XGB 0.590318  0.195073
      LGBM 0.594550  0.189303
       KNN 0.648370  0.115917
       SVR 0.816414 -0.113219
       MLP 1.422580 -0.939755

Best model: CatBoost
[Plot] Saved loss curve → plots/loss_curve_CatBoost.png
[Plot] Saved preds vs actual → plots/preds_vs_actual_CatBoost.png
[Plot] Saved residuals time series → plots/residuals_time_CatBoost.png
[Plot] Saved residual histogram → plots/error_hist_CatBoost.png
[Plot] Saved regression confusion matrix → plots/regression_confusion_CatBoost.png
[Plot] Saved feature importance → plots/feature_importance_CatBoost.png
